[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/14xp/iliad-comp-mech-materials/blob/main/exercises/colab/part1_sequence_probabilities_solutions_colab.ipynb)

> **Running on Colab.** This notebook is self-contained: the **Setup** cells below write the helper modules (`processes`, `solutions`, `tests`, `plotting`) into the session, then import them — nothing else to download. Just run the cells top to bottom. (To open it: *File → Upload notebook*, or from Google Drive / a GitHub link.)

# Part 1: Sequence Probabilities & Next-Token Distributions

A **hidden Markov model** (HMM) generates a stream of observable symbols while transitioning among a set of hidden states you never see directly. We package the whole process into a single **observation-indexed transition tensor** $T$ of shape `(n_obs, n_states, n_states)`:

$$
T[x, i, j] = P(\text{emit symbol } x,\ \text{state } j \; | \; \text{state } i),
$$

and an **initial distribution vector** $\eta^{(\emptyset)}$ of shape `(n_states,)`:

$$
\eta^{(\emptyset)}[i] = P(\text{state } i).
$$

Probabilities are assigned to sequences $x_{1:n} = (x_1, x_2, \ldots, x_n)$ via the expression

$$
P(x_{1:n}) = \eta^{(\emptyset)}T[x_1]T[x_2]\cdots T[x_n] \mathbf{1}
$$

where $\mathbf{1}$ is the all-ones column vector i.e., $\mathbf{1}_i = 1$ for all $i$. 

From sequence probabilities everything else follows. The next-token distribution after a prefix $x_{1:n}$ is the ratio of joints,

$$
P(x_{n+1} = k \mid x_{1:n}) = \frac{P(x_{1:n}, k)}{P(x_{1:n})} = \frac{\eta^{(\emptyset)}T[x_1]T[x_2]\cdots T[x_n]T[k] \mathbf{1}}{\eta^{(\emptyset)}T[x_1]T[x_2]\cdots T[x_n] \mathbf{1}},
$$

which is well-defined exactly when the prefix has nonzero probability.

## What you'll implement

In this part you build the forward machinery from the bottom up, in order:

1. `validate` — assert the process is well-formed (stochastic transition matrix and initial vector).
2. `_propagate` — multiply the `initial_vect` by a sequence of transition matrices.
3. `sequence_probability` — the probability of sampling a sequence from the HMM.
4. `conditional_next_token_probabilities` — the next-token distribution over symbols `(n_obs,)`, conditional on a sequence prefix.
5. `all_next_token_probabilities` — every reachable prefix up to a given depth mapped to its next-token distribution, built breadth-first with zero-probability branches pruned.

## The monkeypatch workflow

You're given an `HMM` skeleton with just an `__init__` (it stores `transition_tensor` and `initial_vect`). Each exercise asks you to define a standalone function, attach it to the class, then run the matching test:

```python
def _propagate(self, sequence, vect=None):
    ...

HMM._propagate = _propagate
tests.test_propagate(HMM)
```

Defining methods this way lets each exercise stand on its own while accumulating onto the same class. Later methods may call ones you defined earlier, so keep them attached as you go.

## Example processes

Two processes from `processes.py` serve as running fixtures:

- **Z1R** (Zero-One-Random): 3 hidden states, **2 symbols**. Emits `0` then `1` deterministically, then a random bit. A clean sanity check with sharp, mostly-deterministic structure.
- **Mess3**: 3 hidden states, **3 symbols**. A symmetric process whose belief states trace out a fractal in the 2-simplex — a richer testbed for the distributions you compute here and the belief-state geometry explored later.

## Setup

These cells write the helper modules into the Colab session and import them. Run them once, top to bottom.

In [ ]:
%%writefile processes.py
"""Hidden-Markov processes used as fixtures/demos in the HMM exercises.

Each process is given as an observation-indexed transition tensor of shape
(n_obs, n_states, n_states): ``tensor[x, i, j]`` is the probability of emitting
symbol ``x`` and moving from hidden state ``i`` to hidden state ``j``. Summing
over the symbol axis gives a row-stochastic state-transition matrix.
"""

import numpy as np


def arch(a: float) -> np.ndarray:
    """Creates a transition matrix for the Arch Process."""
    assert 0 <= a <= 1, f"a must be in [0, 1], got {a}"
    b = (1 - a) / 3
    return np.array(
        [
            [
                [0.8 * a, 0.0, 0.0, 0.0],
                [0.0, 0.2 * a, 0.0, 0.0],
                [0.0, 0.0, 0.4 * a, 0.0],
                [0.0, 0.0, 0.0, 0.6 * a],
            ],
            [
                [0.0, 0.0, 0.0, 0.0],
                [0.0, 0.4 * a, 0.0, 0.4 * b],
                [0.0, 0.0, 0.3 * a, 0.0],
                [0.0, 0.0, 0.0, 0.16 * a],
            ],
            [
                [0.2 * a, b, b, b],
                [b, 0.4 * a, b, 0.6 * b],
                [b, b, 0.3 * a, b],
                [b, b, b, 0.24 * a],
            ],
        ]
    )

def z1r(p: float = 0.5) -> np.ndarray:
    """Zero-One-Random (Z1R) process: 3 hidden states, 2 symbols.

    Emits 0 then 1 deterministically, then a random bit with P(1) = ``p``.
    Stationary distribution is uniform, [1/3, 1/3, 1/3].
    """
    q = 1 - p
    return np.array(
        [
            [[0, 1, 0], [0, 0, 0], [q, 0, 0]],
            [[0, 0, 0], [0, 0, 1], [p, 0, 0]],
        ],
        dtype=float,
    )


def mess3(x: float = 0.15, a: float = 0.2) -> np.ndarray:
    """Mess3 process: 3 hidden states, 3 symbols.

    A symmetric process whose belief states trace out a fractal in the
    2-simplex. Stationary distribution is uniform, [1/3, 1/3, 1/3].
    """
    b = (1 - a) / 2
    y = 1 - 2 * x
    ay, bx, by, ax = a * y, b * x, b * y, a * x
    return np.array(
        [
            [[ay, bx, bx], [ax, by, bx], [ax, bx, by]],
            [[by, ax, bx], [bx, ay, bx], [bx, ax, by]],
            [[by, bx, ax], [bx, by, ax], [bx, bx, ay]],
        ],
        dtype=float,
    )


def uniform_initial(n_states: int) -> np.ndarray:
    """Uniform initial state distribution over ``n_states`` hidden states."""
    return np.ones(n_states) / n_states

In [ ]:
%%writefile solutions.py
"""Reference HMM implementation for the exercises.

The exercise notebooks ask the reader to
reconstruct these methods; the tests in ``tests.py`` compare the reader's work
against this reference.
"""

import numpy as np
from collections.abc import Sequence

type TransitionTensor = np.ndarray  # (n_obs, n_states, n_states)
type StateVector = np.ndarray       # (n_states,) -- belief / propagated state vector
type TokenDist = np.ndarray         # (n_obs,)    -- distribution over next tokens


class HMM:
    """HMM defined by an observation-indexed transition tensor and an initial vector.

    Shapes
    ------
    transition_tensor        : (n_obs, n_states, n_states)
    belief / state vectors   : (n_states,)
    next-token distributions : (n_obs,)
    """

    def __init__(self, transition_tensor: TransitionTensor, initial_vect: StateVector) -> None:
        self.transition_tensor = np.asarray(transition_tensor)
        self.initial_vect = np.atleast_1d(np.asarray(initial_vect)).ravel()

    def validate(self) -> None:
        """Assert that transition_tensor and initial_vect define a valid HMM.

        The observation-summed state-transition matrix must be row-stochastic and
        the initial vector must be a probability distribution. Raises AssertionError
        otherwise.
        """
        assert (self.transition_tensor >= 0).all(), "transition_tensor must be non-negative"
        transition_matrix = self.transition_tensor.sum(axis=0)
        assert np.allclose(transition_matrix.sum(axis=1), 1.0), \
            "summed transition matrix must be row-stochastic (each row sums to 1)"
        assert (self.initial_vect >= 0).all(), "initial_vect must be non-negative"
        assert np.isclose(self.initial_vect.sum(), 1.0), "initial_vect must sum to 1"

    def _propagate(self, sequence: Sequence[int], vect: StateVector | None = None) -> StateVector:
        """Run a sequence through the tensor, returning the unnormalized state vector."""
        vect = self.initial_vect if vect is None else vect
        for obs in sequence:
            vect = np.einsum("i,ij->j", vect, self.transition_tensor[obs])
        return vect

    def sequence_probability(self, sequence: Sequence[int]) -> float:
        return np.sum(self._propagate(sequence))

    def conditional_next_token_probabilities(self, sequence: Sequence[int]) -> TokenDist:
        seq = list(sequence)
        p_seq = self.sequence_probability(seq)
        if np.allclose(p_seq, 0.0):
            raise ValueError(f"sequence {tuple(sequence)} has zero probability; "
                             "next-token distribution is undefined.")
        n_obs = self.transition_tensor.shape[0]
        joint = np.array([self.sequence_probability(seq + [k]) for k in range(n_obs)])
        return joint / p_seq

    def all_next_token_probabilities(self, max_depth: int) -> dict[tuple[int, ...], TokenDist]:
        """Next-token distribution for every reachable sequence of length 0..max_depth.

        Built breadth-first; a continuation is reachable iff its probability in the
        current distribution is nonzero (zero-probability branches are pruned).
        Uses conditional_next_token_probabilities directly -- no belief-state machinery.
        Includes () -> the prior next-token distribution.
        """
        prior = self.conditional_next_token_probabilities(())
        result = {(): prior}
        frontier = [((), prior)]
        for _ in range(max_depth):
            next_frontier = []
            for seq, ntp in frontier:
                for obs in range(len(ntp)):
                    if np.allclose(ntp[obs], 0.0):
                        continue  # zero-probability continuation -> prune
                    new_seq = seq + (obs,)
                    child = self.conditional_next_token_probabilities(new_seq)
                    result[new_seq] = child
                    next_frontier.append((new_seq, child))
            frontier = next_frontier
        return result

    #######################
    # Belief State Methods #
    #######################

    def belief_state(self, sequence: Sequence[int]) -> StateVector:
        vect = self._propagate(sequence)
        norm = np.sum(vect)
        if np.allclose(norm, 0.0):
            raise ValueError(f"sequence {tuple(sequence)} has zero probability; belief state is undefined.")
        return vect / norm

    def belief_update(self, belief_state: StateVector, obs: int) -> StateVector:
        update = np.einsum("i,ij->j", belief_state, self.transition_tensor[obs])
        norm = np.sum(update)
        if np.allclose(norm, 0.0):
            raise ValueError(f"observation {obs} has zero probability from this belief state; cannot update.")
        return update / norm

    def ntp_from_belief_state(self, belief_state: StateVector) -> TokenDist:
        return np.einsum("i,kij->k", belief_state, self.transition_tensor)

    def all_belief_states(self, max_depth: int) -> dict[tuple[int, ...], StateVector]:
        """Belief state for every reachable sequence of length 0..max_depth.

        Returns a dict mapping each observation sequence (as a tuple) to its
        belief state, starting from the empty sequence () -> prior belief.
        Built breadth-first via belief_update; zero-probability continuations
        are pruned (they have no defined belief state).
        """
        n_obs = self.transition_tensor.shape[0]
        prior = self.belief_state(())
        result = {(): prior}
        frontier = [((), prior)]
        for _ in range(max_depth):
            next_frontier = []
            for seq, belief in frontier:
                for obs in range(n_obs):
                    try:
                        updated = self.belief_update(belief, obs)
                    except ValueError:
                        continue  # zero-probability continuation -> prune
                    new_seq = seq + (obs,)
                    result[new_seq] = updated
                    next_frontier.append((new_seq, updated))
            frontier = next_frontier
        return result

In [ ]:
%%writefile tests.py
"""ARENA-style tests for the HMM exercises.

Each ``test_<method>(HMM)`` takes the reader's ``HMM`` class (with the relevant
method monkeypatched on) and checks it against the reference implementation in
``solutions.py``, across the Z1R and Mess3 processes. Inputs to the
method-under-test are always built from the *reference*, so each test isolates a
single method. Prints "All tests passed!" on success; raises ``AssertionError``
(or surfaces an unexpected exception) otherwise.
"""

from itertools import product

import numpy as np

import processes
import solutions


def _fixtures():
    """(label, transition_tensor, initial_vect) cases covering 2- and 3-symbol processes."""
    return [
        ("Z1R", processes.z1r(0.5), processes.uniform_initial(3)),
        ("Mess3", processes.mess3(0.15, 0.2), processes.uniform_initial(3)),
    ]


def _all_seqs(n_obs, max_len=4):
    seqs = [()]
    for length in range(1, max_len + 1):
        seqs.extend(product(range(n_obs), repeat=length))
    return seqs


def _reachable_seqs(ref, n_obs, max_len=4):
    return [s for s in _all_seqs(n_obs, max_len)
            if ref.sequence_probability(list(s)) > 1e-12]


def _first_zero_prob_seq(ref, n_obs, max_len=4):
    for s in _all_seqs(n_obs, max_len):
        if s and np.isclose(ref.sequence_probability(list(s)), 0.0):
            return s
    raise RuntimeError("no zero-probability sequence found for this fixture")


# ----------------------------------------------------------------------------
# Part 1: sequence probabilities / next-token distributions
# ----------------------------------------------------------------------------

def test_validate(HMM):
    # well-formed processes must pass
    for label, tensor, init in _fixtures():
        HMM(tensor, init).validate()
    # each malformed case must raise AssertionError
    good_t, good_i = processes.z1r(0.5), processes.uniform_initial(3)
    neg_t = good_t.copy()
    neg_t[0, 2, 0] = -neg_t[0, 2, 0] - 0.1
    bad_cases = [
        ("non-row-stochastic transition matrix", good_t * 2.0, good_i),
        ("initial_vect that does not sum to 1", good_t, np.ones(3)),
        ("negative initial_vect", good_t, np.array([1.5, -0.5, 0.0])),
        ("negative transition_tensor entry", neg_t, good_i),
    ]
    for desc, tensor, init in bad_cases:
        try:
            HMM(tensor, init).validate()
        except AssertionError:
            pass
        else:
            raise AssertionError(f"validate() should have rejected: {desc}")
    print("All tests passed!")


def test_propagate(HMM):
    for label, tensor, init in _fixtures():
        ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
        n_obs = tensor.shape[0]
        for s in _all_seqs(n_obs):
            expected = ref._propagate(list(s))
            actual = cand._propagate(list(s))
            assert np.shape(actual) == np.shape(expected), (
                f"[{label}] _propagate{tuple(s)} has shape {np.shape(actual)}, "
                f"expected {np.shape(expected)}")
            assert np.allclose(actual, expected), (
                f"[{label}] _propagate{tuple(s)} = {actual}, expected {expected}")
        # passing an explicit `vect` should start from that vector, not the initial one
        custom = np.array([1.0, 0.0, 0.0])
        assert np.allclose(cand._propagate([0], vect=custom),
                           ref._propagate([0], vect=custom)), (
            f"[{label}] _propagate with explicit vect argument is incorrect")
    print("All tests passed!")


def test_sequence_probability(HMM):
    for label, tensor, init in _fixtures():
        ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
        n_obs = tensor.shape[0]
        for s in _all_seqs(n_obs):
            expected = ref.sequence_probability(list(s))
            actual = cand.sequence_probability(list(s))
            assert np.isclose(actual, expected), (
                f"[{label}] sequence_probability{tuple(s)} = {actual}, expected {expected}")
        # proper distribution: probabilities over all length-L sequences sum to 1
        for length in range(1, 5):
            total = sum(cand.sequence_probability(list(s))
                        for s in product(range(n_obs), repeat=length))
            assert np.isclose(total, 1.0), (
                f"[{label}] sum of P(length-{length} sequences) = {total}, expected 1")
    print("All tests passed!")


def test_conditional_next_token_probabilities(HMM):
    for label, tensor, init in _fixtures():
        ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
        n_obs = tensor.shape[0]
        for s in _reachable_seqs(ref, n_obs):
            expected = ref.conditional_next_token_probabilities(s)
            actual = cand.conditional_next_token_probabilities(s)
            assert np.allclose(actual, expected), (
                f"[{label}] conditional_next_token_probabilities{tuple(s)} = {actual}, "
                f"expected {expected}")
            assert np.isclose(np.sum(actual), 1.0), (
                f"[{label}] NTP{tuple(s)} sums to {np.sum(actual)}, expected 1")
    # a zero-probability sequence must raise ValueError (Z1R has unreachable sequences)
    tensor, init = processes.z1r(0.5), processes.uniform_initial(3)
    ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
    zero_seq = _first_zero_prob_seq(ref, 2)
    try:
        cand.conditional_next_token_probabilities(zero_seq)
    except ValueError:
        pass
    else:
        raise AssertionError(
            f"expected ValueError for zero-probability sequence {zero_seq}")
    print("All tests passed!")


def test_all_next_token_probabilities(HMM):
    for label, tensor, init in _fixtures():
        ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
        expected = ref.all_next_token_probabilities(4)
        actual = cand.all_next_token_probabilities(4)
        assert () in actual, f"[{label}] missing the empty-sequence () entry"
        assert set(actual) == set(expected), (
            f"[{label}] sequence key sets differ: "
            f"extra={set(actual) - set(expected)}, missing={set(expected) - set(actual)}")
        for k in expected:
            assert np.allclose(actual[k], expected[k]), (
                f"[{label}] all_next_token_probabilities[{k}] = {actual[k]}, "
                f"expected {expected[k]}")
    print("All tests passed!")


# ----------------------------------------------------------------------------
# Part 2: belief states
# ----------------------------------------------------------------------------

def test_belief_state(HMM):
    for label, tensor, init in _fixtures():
        ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
        n_obs = tensor.shape[0]
        for s in _reachable_seqs(ref, n_obs):
            expected = ref.belief_state(s)
            actual = cand.belief_state(s)
            assert np.allclose(actual, expected), (
                f"[{label}] belief_state{tuple(s)} = {actual}, expected {expected}")
            assert np.isclose(np.sum(actual), 1.0), (
                f"[{label}] belief_state{tuple(s)} sums to {np.sum(actual)}, expected 1")
    # a zero-probability sequence must raise ValueError
    tensor, init = processes.z1r(0.5), processes.uniform_initial(3)
    ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
    zero_seq = _first_zero_prob_seq(ref, 2)
    try:
        cand.belief_state(zero_seq)
    except ValueError:
        pass
    else:
        raise AssertionError(
            f"expected ValueError for zero-probability sequence {zero_seq}")
    print("All tests passed!")


def test_belief_update(HMM):
    for label, tensor, init in _fixtures():
        ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
        n_obs = tensor.shape[0]
        for s in _reachable_seqs(ref, n_obs, max_len=3):
            belief = ref.belief_state(s)
            reachable = ref.ntp_from_belief_state(belief)
            for obs in range(n_obs):
                if reachable[obs] <= 1e-12:
                    continue
                expected = ref.belief_update(belief, obs)
                actual = cand.belief_update(belief, obs)
                assert np.allclose(actual, expected), (
                    f"[{label}] belief_update(belief_state{tuple(s)}, {obs}) = {actual}, "
                    f"expected {expected}")
                assert np.isclose(np.sum(actual), 1.0), (
                    f"[{label}] belief_update result sums to {np.sum(actual)}, expected 1")
    # an impossible observation from a belief state must raise ValueError (Z1R)
    tensor, init = processes.z1r(0.5), processes.uniform_initial(3)
    ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
    found = False
    for s in _reachable_seqs(ref, 2, max_len=3):
        belief = ref.belief_state(s)
        reachable = ref.ntp_from_belief_state(belief)
        for obs in range(2):
            if reachable[obs] <= 1e-12:
                try:
                    cand.belief_update(belief, obs)
                except ValueError:
                    found = True
                else:
                    raise AssertionError(
                        f"expected ValueError for impossible observation {obs} "
                        f"from belief_state{tuple(s)}")
                break
        if found:
            break
    assert found, "could not find an impossible (belief, obs) pair to exercise the guard"
    print("All tests passed!")


def test_ntp_from_belief_state(HMM):
    for label, tensor, init in _fixtures():
        ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
        n_obs = tensor.shape[0]
        for s in _reachable_seqs(ref, n_obs):
            belief = ref.belief_state(s)
            expected = ref.ntp_from_belief_state(belief)
            actual = cand.ntp_from_belief_state(belief)
            assert np.allclose(actual, expected), (
                f"[{label}] ntp_from_belief_state(belief_state{tuple(s)}) = {actual}, "
                f"expected {expected}")
            assert np.isclose(np.sum(actual), 1.0), (
                f"[{label}] next-token distribution sums to {np.sum(actual)}, expected 1")
    print("All tests passed!")


def test_all_belief_states(HMM):
    for label, tensor, init in _fixtures():
        ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
        expected = ref.all_belief_states(4)
        actual = cand.all_belief_states(4)
        assert () in actual, f"[{label}] missing the empty-sequence () entry"
        assert set(actual) == set(expected), (
            f"[{label}] sequence key sets differ: "
            f"extra={set(actual) - set(expected)}, missing={set(expected) - set(actual)}")
        for k in expected:
            assert np.allclose(actual[k], expected[k]), (
                f"[{label}] all_belief_states[{k}] = {actual[k]}, expected {expected[k]}")
    print("All tests passed!")

In [ ]:
%%writefile plotting.py
"""Interactive simplex plotting for the HMM exercises.

Self-contained — only numpy + plotly are required; scikit-learn is
imported lazily and only used for the >4-dimensional PCA path (the example
processes here are 3-4 dimensional, so it is never needed).
"""

import numpy as np
import plotly.graph_objects as go


def regular_simplex_vertices(n: int) -> np.ndarray:
    """Vertices of a regular (n-1)-simplex, centered, as an (n, n-1) array.

    The n probability-simplex corners e_i live in the hyperplane sum=1; centering
    them and projecting onto an orthonormal basis of the ones-orthogonal-complement
    yields a regular simplex in R^(n-1) (equilateral triangle for n=3, regular
    tetrahedron for n=4). Orientation is arbitrary -- only used for visualization.
    """
    centered = np.eye(n) - 1.0 / n
    # Rows of `centered` span the (n-1)-dim subspace orthogonal to the all-ones
    # vector; Vt[:n-1] is an orthonormal basis of it.
    _, _, vt = np.linalg.svd(centered)
    return centered @ vt[: n - 1].T


def _coords_to_rgb(beliefs: np.ndarray) -> list:
    """Map each belief vector to an RGB color via its first up-to-3 coordinates.

    For a 3-state model this is the exact barycentric coloring (each simplex
    corner -> pure red/green/blue); for more states only the first 3 coordinates
    drive the color.
    """
    rgb = np.zeros((len(beliefs), 3))
    k = min(3, beliefs.shape[1])
    rgb[:, :k] = np.clip(beliefs[:, :k], 0.0, 1.0)
    return ["rgb(%d,%d,%d)" % tuple((c * 255).astype(int)) for c in rgb]


def plot_belief_states(beliefs, sequences=None, path=None, title=None,
                       color_by_coords=True):
    """Plot HMM belief states on the hidden-state simplex (see plot_simplex_points)."""
    n = np.asarray(beliefs).shape[1]
    if title is None:
        title = f"Belief states ({n} hidden states)"
    return plot_simplex_points(beliefs, sequences, path, title, color_by_coords)


def plot_next_token_distributions(ntps, sequences=None, path=None,
                                  title=None, color_by_coords=True):
    """Plot next-token distributions on the symbol simplex (see plot_simplex_points)."""
    n = np.asarray(ntps).shape[1]
    if title is None:
        title = f"Next-token distributions ({n} symbols)"
    return plot_simplex_points(ntps, sequences, path, title, color_by_coords)


def plot_simplex_points(points, sequences=None, path=None, title=None,
                        color_by_coords=True):
    """Build an interactive Plotly scatter of probability vectors on a simplex.

    points: (N, d) array of probability vectors (complex inputs, e.g. GHMM
        beliefs/NTPs, are cast to their real part).
    sequences: optional length-N list of the observation sequences that induced
        each point; shown on hover. If None, hover shows the point index.
    path: if given, also write the figure to this HTML file. The Plotly figure is
        always returned (renders inline in a notebook).
    color_by_coords: if True, color each point by its coordinates (RGB from the
        first up-to-3 components); for d == 3 this is the exact barycentric coloring.

    The points are projected onto the probability simplex, then:
      - d == 2 -> 1D strip (x-axis)
      - d == 3 -> 2D triangle
      - d == 4 -> 3D tetrahedron
      - d  > 4 -> PCA to the top 3 components, 3D scatter
    """
    beliefs = np.real(np.asarray(points, dtype=complex))
    n_points, n = beliefs.shape
    if n < 2:
        raise ValueError(f"need at least 2 simplex dimensions to plot, got {n}.")
    if title is None:
        title = f"Simplex points ({n}-simplex)"

    if sequences is not None:
        if len(sequences) != n_points:
            raise ValueError(
                f"sequences has length {len(sequences)} but there are {n_points} points."
            )
        hover = [f"input sequence: {tuple(s)}" if len(tuple(s)) else "input sequence: () [start]"
                 for s in sequences]
    else:
        hover = [f"index {i}" for i in range(n_points)]

    coords = beliefs @ regular_simplex_vertices(n)

    marker = dict(size=5, opacity=0.7)
    if color_by_coords:
        marker["color"] = _coords_to_rgb(beliefs)
    hovertemplate = "%{text}<extra></extra>"

    if n == 2:
        fig = go.Figure(
            go.Scatter(
                x=coords[:, 0], y=np.zeros(n_points), mode="markers",
                marker=marker, text=hover, hovertemplate=hovertemplate,
            )
        )
        fig.update_yaxes(visible=False)

    elif n == 3:
        verts = regular_simplex_vertices(3)
        loop = np.vstack([verts, verts[0]])  # close the triangle
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=loop[:, 0], y=loop[:, 1], mode="lines",
            line=dict(color="black"), hoverinfo="skip", showlegend=False,
        ))
        fig.add_trace(go.Scatter(
            x=coords[:, 0], y=coords[:, 1], mode="markers",
            marker=marker, text=hover, hovertemplate=hovertemplate, showlegend=False,
        ))
        fig.update_yaxes(scaleanchor="x", scaleratio=1, autorange="reversed")

    elif n == 4:
        verts = regular_simplex_vertices(4)
        edges = [(i, j) for i in range(4) for j in range(i + 1, 4)]
        ex, ey, ez = [], [], []
        for i, j in edges:
            ex += [verts[i, 0], verts[j, 0], None]
            ey += [verts[i, 1], verts[j, 1], None]
            ez += [verts[i, 2], verts[j, 2], None]
        fig = go.Figure()
        fig.add_trace(go.Scatter3d(
            x=ex, y=ey, z=ez, mode="lines",
            line=dict(color="black"), hoverinfo="skip", showlegend=False,
        ))
        fig.add_trace(go.Scatter3d(
            x=coords[:, 0], y=coords[:, 1], z=coords[:, 2], mode="markers",
            marker=marker, text=hover, hovertemplate=hovertemplate, showlegend=False,
        ))
        fig.update_scenes(yaxis_autorange="reversed")

    else:  # n > 4: PCA to 3D
        from sklearn.decomposition import PCA

        coords = PCA(n_components=3).fit_transform(coords)
        fig = go.Figure(go.Scatter3d(
            x=coords[:, 0], y=coords[:, 1], z=coords[:, 2], mode="markers",
            marker=marker, text=hover, hovertemplate=hovertemplate,
        ))
        fig.update_scenes(
            xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"
        )

    fig.update_layout(title=title)
    if path is not None:
        fig.write_html(path)
    return fig

In [ ]:
# Best-effort auto-reload of the helper modules written above (silently skipped
# where IPython's autoreload extension is unavailable, e.g. some Colab 3.12 images).
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

import numpy as np
from collections.abc import Sequence

import processes
import solutions
import tests

# Shape aliases (documentation only).
type TransitionTensor = np.ndarray  # (n_obs, n_states, n_states)
type StateVector = np.ndarray       # (n_states,) -- belief / propagated state vector
type TokenDist = np.ndarray         # (n_obs,)    -- distribution over next tokens

---

## Exercises

The cells above are **setup**. First run the cell below to define the `HMM` class, then work through the exercises in order: for each, complete the `# YOUR CODE HERE` cell (it attaches the method to `HMM` via `HMM.<name> = <name>`), then run its `tests.test_<name>(HMM)` cell — it prints **All tests passed!** when your implementation is correct.

The `HMM` skeleton — each exercise adds a method to this class via `HMM.<name> = <name>`.

In [ ]:
class HMM:
    """HMM defined by an observation-indexed transition tensor and an initial vector.

    Shapes
    ------
    transition_tensor        : (n_obs, n_states, n_states)
    belief / state vectors   : (n_states,)
    next-token distributions : (n_obs,)
    """

    def __init__(self, transition_tensor: TransitionTensor, initial_vect: StateVector) -> None:
        self.transition_tensor = np.asarray(transition_tensor)
        self.initial_vect = np.atleast_1d(np.asarray(initial_vect)).ravel()

### Exercise - implement `validate`

> **Difficulty:** 🔴🔴⚪⚪⚪  
> **Importance:** 🔵🔵🔵⚪⚪  
> 
> You should spend up to ~10 minutes on this exercise.

Before computing anything, it is worth checking that `transition_tensor` and `initial_vect` actually define a valid hidden Markov model. Implement `validate`, which `assert`s the structural constraints (and so raises `AssertionError` on a malformed process).

Two conditions must hold:

- **The transition matrix is stochastic.** Every entry of $T$ is a probability, so $T \ge 0$. Summing over the symbol axis collapses the emissions into the underlying state-transition matrix $T_{ij} = \sum_x T[x, i, j] = P(j \;|\; i)$, which must be **row-stochastic**: each row sums to $1$ (from any state, *some* (symbol, next-state) pair occurs with total probability $1$).

- **The initial vector is stochastic.** $\text{initial\_vect} \ge 0$ and $\sum_i \text{initial\_vect}_i = 1$.

Write one `assert` per condition with an informative message; use `np.allclose` / `np.isclose` so floating-point round-off does not trip the checks.

<details><summary>Hint</summary>

`self.transition_tensor.sum(axis=0)` is the state-transition matrix $T$; its row sums are `T.sum(axis=1)`. The four checks are `(self.transition_tensor >= 0).all()`, `np.allclose(T.sum(axis=1), 1.0)`, `(self.initial_vect >= 0).all()`, and `np.isclose(self.initial_vect.sum(), 1.0)`.

</details>

In [ ]:
def validate(self) -> None:
    """Assert that transition_tensor and initial_vect define a valid HMM.

    The observation-summed state-transition matrix must be row-stochastic and
    the initial vector must be a probability distribution. Raises AssertionError
    otherwise.
    """
    assert (self.transition_tensor >= 0).all(), "transition_tensor must be non-negative"
    transition_matrix = self.transition_tensor.sum(axis=0)
    assert np.allclose(transition_matrix.sum(axis=1), 1.0), \
        "summed transition matrix must be row-stochastic (each row sums to 1)"
    assert (self.initial_vect >= 0).all(), "initial_vect must be non-negative"
    assert np.isclose(self.initial_vect.sum(), 1.0), "initial_vect must sum to 1"

HMM.validate = validate

In [ ]:
tests.test_validate(HMM)

### Exercise - implement `_propagate`

> **Difficulty:** 🔴🔴⚪⚪⚪  
> **Importance:** 🔵🔵🔵🔵⚪  
> 
> You should spend up to ~10-15 minutes on this exercise.

This is the workhorse that nearly every other method calls. Given an observation `sequence` $x_1, x_2, \dots, x_L$, it runs the **forward recursion** over hidden states, starting from a row vector (`initial_vect` by default, or a supplied `vect`) and right-multiplying by the observation-conditioned matrix at each step.

Concretely, with `transition_tensor` $T$ of shape `(n_obs, n_states, n_states)` where $T[x, i, j] = P(\text{emit } x,\ j\;|\; i)$, the recursion is

$$
\alpha^{(\emptyset)} = \texttt{vect}, \qquad \alpha^{(x_{1:t})} = \alpha^{(x_{1:t-1})}\, T[x_t], \qquad \text{return } \alpha^{(x_{1:L})}.
$$

The result is the **unnormalized** forward state vector $\alpha^{(x_{1:L})}$ of shape `(n_states,)`. Component $j$ accumulates the total probability mass of all hidden-state paths consistent with the observed prefix that end in state $j$; hence $\sum_j \alpha^{(x_{1:L})}_j = P(x_1, \dots, x_L)$. Do not normalize here — downstream callers rely on the raw mass.

<details><summary>Hint</summary>

Default `vect` to `self.initial_vect` when `None`. Then loop over each `obs` in the sequence, updating `vect` with the row-vector-times-matrix product:

```python
vect = np.einsum("i,ij->j", vect, self.transition_tensor[obs])
```

Return `vect` after the loop (it is $\alpha^{(x_{1:L})}$).

</details>

In [ ]:
def _propagate(self, sequence: Sequence[int], vect: StateVector | None = None) -> StateVector:
    """Run a sequence through the tensor, returning the unnormalized state vector."""
    vect = self.initial_vect if vect is None else vect
    for obs in sequence:
        vect = np.einsum("i,ij->j", vect, self.transition_tensor[obs])
    return vect

HMM._propagate = _propagate

In [ ]:
tests.test_propagate(HMM)

### Exercise - implement `sequence_probability`

> **Difficulty:** 🔴⚪⚪⚪⚪  
> **Importance:** 🔵🔵🔵⚪⚪  
> 
> You should spend up to ~5-10 minutes on this exercise.

This method returns the total probability that the process emits exactly the given observation sequence, $P(x_{1:L})$. With `_propagate` already in hand, you receive the (unnormalized) forward vector $\alpha^{(x_{1:L})} \in \mathbb{R}^{n\_states}$, whose component $\alpha_j^{(x_{1:L})}$ is the joint probability of having emitted $x_{1:L}$ *and* landing in state $j$:

$$
\alpha_j^{(x_{1:L})} = P(x_{1:L},\, s_L = j).
$$

To get the marginal probability of the sequence, sum out the unobserved final state. Summing $\alpha$ over $j$ contracts the forward vector with the all-ones final vector $\mathbf{1}$, which is the correct termination step for a standard HMM (every state is an admissible stopping point):

$$
P(x_{1:L}) = \alpha^{(x_{1:L})} \mathbf{1} = \sum_j \alpha_j^{(x_{1:L})}.
$$

<details><summary>Hint</summary>

Call `_propagate` to obtain $\alpha$, then reduce it to a scalar by summing over the state axis (the contraction with $\mathbf{1}$).

</details>

In [ ]:
def sequence_probability(self, sequence: Sequence[int]) -> float:
    return np.sum(self._propagate(sequence))

HMM.sequence_probability = sequence_probability

In [ ]:
tests.test_sequence_probability(HMM)

### Exercise - implement `conditional_next_token_probabilities`

> **Difficulty:** 🔴⚪⚪⚪⚪  
> **Importance:** 🔵🔵🔵🔵⚪  
> 
> You should spend up to ~5-10 minutes on this exercise.


Given an observed sequence $x_{1:L}$, this returns the distribution over the next symbol, $P(\text{next} = k \mid x_{1:L})$ for each $k \in \{0, \dots, n_{obs}-1\}$. The result is a `TokenDist` of shape `(n_obs,)`.

Apply the definition of conditional probability directly:

$$
P(\text{next} = k \mid x_{1:L}) = \frac{P(x_{1:L},\, k)}{P(x_{1:L})}
$$

Both numerator and denominator are values you can already obtain from `sequence_probability`: the numerator is the probability of the sequence with $k$ appended, and the denominator is the probability of the sequence on its own. No belief states are needed — this is purely a ratio of sequence probabilities.

If $P(x_{1:L}) = 0$ the conditional is undefined (division by zero), so raise a `ValueError` rather than returning anything.

<details><summary>Hint</summary>

No einsum here. Compute the denominator once as `sequence_probability(sequence)`; if it is `0`, raise `ValueError`. Then, for each candidate next symbol $k$ in `range(n_obs)`, evaluate `sequence_probability(sequence + [k])` and divide by the denominator. Collect the `n_obs` ratios into a single vector.

</details>

In [ ]:
def conditional_next_token_probabilities(self, sequence: Sequence[int]) -> TokenDist:
    seq = list(sequence)
    p_seq = self.sequence_probability(seq)
    if np.allclose(p_seq, 0.0):
        raise ValueError(f"sequence {tuple(sequence)} has zero probability; "
                         "next-token distribution is undefined.")
    n_obs = self.transition_tensor.shape[0]
    joint = np.array([self.sequence_probability(seq + [k]) for k in range(n_obs)])
    return joint / p_seq

HMM.conditional_next_token_probabilities = conditional_next_token_probabilities

In [ ]:
tests.test_conditional_next_token_probabilities(HMM)

### Exercise - implement `all_next_token_probabilities`

> **Difficulty:** 🔴🔴🔴⚪⚪  
> **Importance:** 🔵🔵⚪⚪⚪  
> 
> You should spend up to ~15-20 minutes on this exercise.

This method enumerates the next-token distribution for *every reachable* observation sequence up to length `max_depth`, returning a dict keyed by the sequence tuple. The empty sequence `()` maps to the prior next-token distribution `conditional_next_token_probabilities(())`, and a sequence like `(2, 0)` maps to $P(x_3 \mid x_{1:2} = (2, 0))$. Each value is a `TokenDist` of shape `(n_obs,)`.

Proceed breadth-first. Start from the frontier $\{()\}$ with its next-token distribution. From a frontier sequence `seq` with distribution `ntp`, an observation `obs` is a *reachable* continuation iff $\text{ntp}[\text{obs}] > 0$; the conditional distribution of `seq + (obs,)` is then well defined and is obtained from `conditional_next_token_probabilities`. Continuations with $\text{ntp}[\text{obs}] = 0$ have probability zero, so the conditional state is undefined — prune those branches rather than recursing into them.

This is the exhaustive, exact counterpart to sampling: instead of following one trajectory, it expands the full reachable tree of histories, which makes it useful for computing exact statistics (e.g. entropies, myopic predictions) over all paths the process can produce.

<details><summary>Hint</summary>

No einsum here — this is plain tree traversal. Maintain a result dict and a frontier (queue) of sequences whose next-token distributions you have. For each `seq` in the current frontier, store its `ntp` in the result, then if `len(seq) < max_depth`, loop over `obs in range(n_obs)` and enqueue `seq + (obs,)` only when `ntp[obs]` is nonzero, computing its distribution via `self.conditional_next_token_probabilities(seq + (obs,))`. Seed the loop with `seq = ()`.

</details>

In [ ]:
def all_next_token_probabilities(self, max_depth: int) -> dict[tuple[int, ...], TokenDist]:
    """Next-token distribution for every reachable sequence of length 0..max_depth.

    Built breadth-first; a continuation is reachable iff its probability in the
    current distribution is nonzero (zero-probability branches are pruned).
    Uses conditional_next_token_probabilities directly -- no belief-state machinery.
    Includes () -> the prior next-token distribution.
    """
    prior = self.conditional_next_token_probabilities(())
    result = {(): prior}
    frontier = [((), prior)]
    for _ in range(max_depth):
        next_frontier = []
        for seq, ntp in frontier:
            for obs in range(len(ntp)):
                if np.allclose(ntp[obs], 0.0):
                    continue  # zero-probability continuation -> prune
                new_seq = seq + (obs,)
                child = self.conditional_next_token_probabilities(new_seq)
                result[new_seq] = child
                next_frontier.append((new_seq, child))
        frontier = next_frontier
    return result

HMM.all_next_token_probabilities = all_next_token_probabilities

In [ ]:
tests.test_all_next_token_probabilities(HMM)

## Demo

With the methods implemented, we can explore a process. The cells below render the **next-token-distribution geometry** interactively (via the local `plotting.py`): every reachable sequence's next-token distribution, plotted on the symbol simplex and colored by its coordinates. Hover a point to see the sequence that induces it. We show two 3-symbol processes — **Mess3** (3 states; a fractal) and **arch** (4 states).

In [ ]:
from itertools import product

import plotting

hmm = HMM(processes.mess3(0.15, 0.2), processes.uniform_initial(3))
hmm.validate()  # a well-formed process passes silently

# A proper process: probabilities over all length-3 sequences sum to 1.
total = sum(hmm.sequence_probability(s) for s in product(range(3), repeat=3))
print("sum P(length-3 sequences) =", round(float(total), 6))

print("P(next | (0, 1))           =", np.round(hmm.conditional_next_token_probabilities((0, 1)), 4))
print("# reachable seqs <= depth 4:", len(hmm.all_next_token_probabilities(4)))

# Interactive next-token-distribution geometry (uses the local plotting.py).
# Each point is the next-token distribution after some sequence, plotted on the
# symbol simplex and colored by its coordinates; hover shows the inducing sequence.
depth = 7
ntps = hmm.all_next_token_probabilities(depth)
plotting.plot_next_token_distributions(
    np.array([np.real(v) for v in ntps.values()]),
    sequences=list(ntps.keys()),
    title=f"Mess3 next-token-distribution geometry (depth {depth})",
)

In [ ]:
# The arch process: 4 hidden states, 3 symbols. Its next-token distributions still
# live on the 3-symbol simplex (a 2D triangle), but trace a different geometry.
arch_hmm = HMM(processes.arch(0.6), processes.uniform_initial(4))
arch_hmm.validate()

arch_ntps = arch_hmm.all_next_token_probabilities(depth)
plotting.plot_next_token_distributions(
    np.array([np.real(v) for v in arch_ntps.values()]),
    sequences=list(arch_ntps.keys()),
    title=f"Arch next-token-distribution geometry (depth {depth})",
)